In [ ]:
%pip install ipywidgets

In [ ]:
import io, warnings
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy.ndimage import zoom as nd_zoom, rotate as nd_rotate, gaussian_filter
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, Layout, Output
from IPython.display import display, Image, clear_output
warnings.filterwarnings('ignore')

_sl=Layout(width="380px"); _sl2=Layout(width="420px"); _sl3=Layout(width="440px")
_sty={"description_width":"120px"}

def make_letter(text, size=64):
    fig,ax=plt.subplots(figsize=(2,2)); ax.axis('off')
    fig.text(0.23,0.26,text,fontsize=100); fig.canvas.draw()
    buf=np.frombuffer(fig.canvas.buffer_rgba(),dtype=np.uint8)
    w,h=fig.canvas.get_width_height()
    data=buf.reshape(h,w,4)[:,:,:3].mean(2); plt.close(fig)
    data=(data-data.min())/(data.max()-data.min()+1e-10)
    data=(~data.astype(bool))[::-1].astype(float)
    return nd_zoom(data,size/data.shape[0])

def make_circle(size=64,radius_frac=0.35):
    y,x=np.mgrid[0:size,0:size].astype(float)
    return ((x-size/2)**2+(y-size/2)**2<(size*radius_frac)**2).astype(float)

def low_pass_filter(im,sigma=3.0): return gaussian_filter(im,sigma=sigma)

def show(im,ax=None,title='',vmin=None,vmax=None,cmap='gray'):
    if ax is None: _,ax=plt.subplots()
    ax.imshow(im,cmap=cmap,origin='lower',vmin=vmin if vmin is not None else im.min(),vmax=vmax if vmax is not None else im.max())
    ax.axis('off')
    if title: ax.set_title(title,fontsize=9)

def fig2img(fig):
    buf=io.BytesIO(); fig.savefig(buf,format='png',dpi=100,bbox_inches='tight')
    buf.seek(0); plt.close(fig); return Image(data=buf.read())

def simulate_image(image,angle,sigma=1,rng=None):
    rng=rng or np.random.default_rng()
    return nd_rotate(image,angle,reshape=False)+sigma*rng.standard_normal(image.shape)

def simulate_images(image,sigma,N,seed=42):
    rng=np.random.default_rng(seed); angles=rng.uniform(0,360,N)
    return angles,np.array([simulate_image(image,a,sigma,rng) for a in angles])

def rotate_reference(reference,delta_angle=5):
    angles=np.arange(0,360,delta_angle)
    return angles,np.array([nd_rotate(reference,a,reshape=False) for a in angles])

def correlation(image,reference_stack):
    return (image[None]*reference_stack).mean(axis=(1,2))

def align_images(images_stack,reference_stack,cand_angles):
    return np.array([cand_angles[int(correlation(img,reference_stack).argmax())] for img in images_stack])

def reconstruct(images_stack,estimated_angles):
    recon=np.zeros_like(images_stack[0])
    for angle,img in zip(estimated_angles,images_stack): recon+=nd_rotate(img,-angle,reshape=False)
    return recon/len(images_stack)

def translate_image(im,dx,dy): return np.roll(np.roll(im,dy,axis=0),dx,axis=1)

def simulate_images_with_translation(image,sigma,N,max_t=10,seed=42):
    rng=np.random.default_rng(seed); angles=rng.uniform(0,360,N)
    dxs=rng.integers(-max_t,max_t+1,N); dys=rng.integers(-max_t,max_t+1,N)
    imgs=[]
    for ang,dx,dy in zip(angles,dxs,dys):
        rot=nd_rotate(image,ang,reshape=False)
        imgs.append(translate_image(rot,dx,dy)+sigma*rng.standard_normal(image.shape))
    return np.array(imgs),angles,dxs,dys

def build_rotation_translation_stack(reference,delta_angle=10,max_t=8,t_step=4):
    angles=np.arange(0,360,delta_angle); ts=np.arange(-max_t,max_t+1,t_step)
    stack,al,dl,dyl=[],[],[],[]
    for a in angles:
        rot=nd_rotate(reference,a,reshape=False)
        for dx in ts:
            for dy in ts:
                stack.append(translate_image(rot,dx,dy)); al.append(a); dl.append(dx); dyl.append(dy)
    return np.array(stack),np.array(al),np.array(dl),np.array(dyl)

def align_with_translation(images,ref_stack,ang_list,dx_list,dy_list):
    n,h,w=images.shape; R=ref_stack.shape[0]
    A=images.reshape(n,h*w).astype(float); B=ref_stack.reshape(R,h*w).astype(float)
    cc=(A@B.T)/(h*w); best=cc.argmax(1)
    return ang_list[best],dx_list[best],dy_list[best]

def reconstruct_from_rotation_translation(images,est_angles,est_dx,est_dy):
    recon=np.zeros_like(images[0])
    for img,ang,dx,dy in zip(images,est_angles,est_dx,est_dy):
        recon+=nd_rotate(translate_image(img,-dx,-dy),-ang,reshape=False)
    return recon/len(images)

def log_likelihood(image,reference,sigma):
    return -0.5*np.sum((image-reference)**2)/sigma**2

def ml_reconstruct(images,ref_stack,cand_angles,sigma=2.0):
    recon=np.zeros_like(images[0])
    for img in images:
        ll=np.array([log_likelihood(img,ref,sigma) for ref in ref_stack])
        ll-=ll.max(); w=np.exp(ll); w/=w.sum()
        for wi,ang in zip(w,cand_angles): recon+=wi*nd_rotate(img,-ang,reshape=False)
    return recon/len(images)

# Pre-computed
Q=make_letter('Q',size=64)
_REFS={'Q (oracle)':Q,'Blurred Q (σ=5)':low_pass_filter(Q,sigma=5),
       'Circle':make_circle(64),'P (wrong)':make_letter('P',64),'O (wrong)':make_letter('O',64)}
_CAND5=np.arange(0,360,5)
_STACKS5={name:np.array([nd_rotate(ref,a,reshape=False) for a in _CAND5]) for name,ref in _REFS.items()}
true_angles_ex,images_ex=simulate_images(Q,2.0,200,seed=42)
_ref_ex=low_pass_filter(Q,sigma=5); _ca_ex,_rs_ex=rotate_reference(_ref_ex,delta_angle=5)
estimated_angles_ex=align_images(images_ex,_rs_ex,_ca_ex)
_CLS_LETTERS=['Q','P','R']; _CLS_IMGS={lt:make_letter(lt,64) for lt in _CLS_LETTERS}
_CLS_CAND=np.arange(0,360,10)
_CLS_STACKS={lt:np.array([nd_rotate(_CLS_IMGS[lt],a,reshape=False) for a in _CLS_CAND]) for lt in _CLS_LETTERS}

# ── Display functions ─────────────────────────────────────────────────────────
def show_test_image():
    fig,ax=plt.subplots(figsize=(3,3)); show(Q,ax=ax,title="Test image: Q (64×64)"); display(fig2img(fig))

def single_image():
    angle_sl=IntSlider(value=30,min=0,max=359,step=1,description="Angle (°)",style=_sty,layout=_sl)
    sigma_sl=FloatSlider(value=1.0,min=0.0,max=8.0,step=0.5,description="Noise σ",style=_sty,layout=_sl)
    out=Output()
    def _u(_=None):
        img=simulate_image(Q,angle_sl.value,sigma_sl.value,np.random.default_rng(7))
        fig,axes=plt.subplots(1,3,figsize=(10,3.5))
        show(Q,axes[0],title="True structure A")
        show(nd_rotate(Q,angle_sl.value,reshape=False),axes[1],title=f"Rotated: {angle_sl.value}°")
        v=max(abs(img.min()),abs(img.max())); show(img,axes[2],vmin=-v,vmax=v,title=f"Observed: θ={angle_sl.value}°, σ={sigma_sl.value:.1f}")
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [angle_sl,sigma_sl]: w.observe(_u,names='value')
    display(VBox([angle_sl,sigma_sl,out])); _u()

def reference_selection():
    ref_dd=Dropdown(options=list(_REFS.keys()),value='Blurred Q (σ=5)',description="Reference:",style=_sty)
    sig_sl=FloatSlider(value=2.0,min=0.0,max=8.0,step=0.5,description="Noise σ",style=_sty,layout=_sl2)
    out=Output()
    def _u(_=None):
        img=simulate_image(Q,130,sig_sl.value,np.random.default_rng(3))
        ref=_REFS[ref_dd.value]; stack=_STACKS5[ref_dd.value]; scores=correlation(img,stack); best=_CAND5[scores.argmax()]
        fig,axes=plt.subplots(1,4,figsize=(14,3.5))
        show(ref,axes[0],title=f"Reference\n({ref_dd.value})")
        v=max(abs(img.min()),abs(img.max())); show(img,axes[1],vmin=-v,vmax=v,title=f"Noisy image (σ={sig_sl.value:.1f})\ntrue: 130°")
        axes[2].plot(_CAND5,scores,color='steelblue',lw=1.4)
        axes[2].axvline(130,color='red',lw=1.4,ls='--',label='True 130°'); axes[2].axvline(best,color='orange',lw=1.4,ls=':',label=f'Best {best}°')
        axes[2].set_xlabel("Candidate angle (°)"); axes[2].set_ylabel("Correlation"); axes[2].set_title("Alignment score"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
        show(nd_rotate(img,-best,reshape=False),axes[3],title=f"Back-rotated {best}°")
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [ref_dd,sig_sl]: w.observe(_u,names='value')
    display(VBox([ref_dd,sig_sl,out])); _u()

def alignment_demo():
    err=np.abs(((estimated_angles_ex-true_angles_ex+180)%360)-180)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].scatter(true_angles_ex,estimated_angles_ex,s=2,c='steelblue',alpha=0.6); axes[0].plot([0,360],[0,360],'r--',lw=1)
    axes[0].set_xlabel("True angle (°)"); axes[0].set_ylabel("Estimated (°)"); axes[0].set_title("Angle estimation (N=200, σ=2)"); axes[0].set_xlim(0,360); axes[0].set_ylim(0,360)
    axes[1].hist(err,bins=36,color='steelblue',edgecolor='k',lw=0.5); axes[1].set_xlabel("Angular error (°)"); axes[1].set_ylabel("Count"); axes[1].set_title(f"Median error: {np.median(err):.1f}°")
    plt.tight_layout(); display(fig2img(fig))

def reconstruction_result():
    recon_ex=reconstruct(images_ex,estimated_angles_ex)
    fig,axes=plt.subplots(1,3,figsize=(10,3.5))
    show(Q,axes[0],title="True structure A"); show(images_ex[0],axes[1],vmin=images_ex[0].min(),vmax=images_ex[0].max(),title="Single image (σ=2.0)")
    show(recon_ex,axes[2],title="Reconstruction (N=200)"); plt.tight_layout(); display(fig2img(fig))

def reconstruction_explorer():
    n_sl=IntSlider(  value=100,min=10, max=500,step=10, description="# images",  style=_sty,layout=_sl3)
    s_sl=FloatSlider(value=2.0,min=0.0,max=10.0,step=0.5,description="Noise σ",   style=_sty,layout=_sl3)
    b_sl=FloatSlider(value=5.0,min=0.0,max=15.0,step=1.0,description="LP blur σ", style=_sty,layout=_sl3)
    d_sl=IntSlider(  value=5,  min=1,  max=20, step=1,  description="Δangle (°)", style=_sty,layout=_sl3)
    out=Output()
    def _u(_=None):
        ta,imgs=simulate_images(Q,s_sl.value,n_sl.value,seed=42)
        ref=low_pass_filter(Q,sigma=b_sl.value) if b_sl.value>0 else Q.copy()
        ca,rs=rotate_reference(ref,delta_angle=d_sl.value); ea=align_images(imgs,rs,ca)
        recon=reconstruct(imgs,ea); qc=float((recon*Q).mean()/(np.sqrt((recon**2).mean()*(Q**2).mean())+1e-10))
        err=np.abs(((ea-ta+180)%360)-180)
        fig,axes=plt.subplots(1,4,figsize=(16,3.8))
        show(ref,axes[0],title="Initial reference"); show(recon,axes[1],title=f"Reconstruction (CC={qc:.2f})")
        axes[2].scatter(ta,ea,s=1,c='steelblue',alpha=0.5); axes[2].plot([0,360],[0,360],'r--',lw=1)
        axes[2].set_xlabel("True angle (°)"); axes[2].set_ylabel("Estimated (°)"); axes[2].set_title(f"Accuracy (med. err={np.median(err):.0f}°)")
        axes[3].hist(err,bins=36,color='steelblue',ec='k',lw=0.4); axes[3].set_xlabel("Error (°)"); axes[3].set_ylabel("Count")
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [n_sl,s_sl,b_sl,d_sl]: w.observe(_u,names='value')
    display(VBox([n_sl,s_sl,b_sl,d_sl,out])); _u()

def translation_demo():
    imgs_t,ang_t,dx_t,dy_t=simulate_images_with_translation(Q,sigma=1.0,N=6,max_t=15,seed=7)
    fig,axes=plt.subplots(1,7,figsize=(16,2.8)); show(Q,axes[0],title="True A")
    for i in range(6):
        axes[i+1].imshow(imgs_t[i],cmap='gray',origin='lower')
        axes[i+1].set_title(f"θ={ang_t[i]:.0f}°\ndx={dx_t[i]},dy={dy_t[i]}",fontsize=7); axes[i+1].axis('off')
    plt.suptitle("Images with rotation AND translation",fontsize=9); plt.tight_layout(); display(fig2img(fig))

def translation_alignment():
    N_t=80; imgs_t,ta_t,tdx,tdy=simulate_images_with_translation(Q,1.0,N_t,max_t=10,seed=42)
    ref_t=low_pass_filter(Q,sigma=5); st,al,dl,dyl=build_rotation_translation_stack(ref_t,delta_angle=10,max_t=10,t_step=4)
    ea_t,edx_t,edy_t=align_with_translation(imgs_t,st,al,dl,dyl)
    recon_t=reconstruct_from_rotation_translation(imgs_t,ea_t,edx_t,edy_t)
    fig,axes=plt.subplots(1,3,figsize=(10,3.5))
    show(Q,axes[0],title="True A"); show(imgs_t[0],axes[1],vmin=imgs_t[0].min(),vmax=imgs_t[0].max(),title="Sample image (σ=1.0)")
    show(recon_t,axes[2],title=f"Reconstruction with translation (N={N_t})"); plt.tight_layout(); display(fig2img(fig))

def ml_comparison():
    ref_ml=low_pass_filter(Q,sigma=5); c_ml,s_ml=rotate_reference(ref_ml,delta_angle=10)
    fig,axes=plt.subplots(2,5,figsize=(16,7))
    for col,sig in enumerate([0.5,1.5,3.0,5.0,8.0]):
        ta,imgs=simulate_images(Q,sig,60,seed=42); eh=align_images(imgs,s_ml,c_ml)
        rh=reconstruct(imgs,eh); rs=ml_reconstruct(imgs,s_ml,c_ml,sigma=sig)
        axes[0,col].imshow(rh,cmap='gray',origin='lower'); axes[0,col].axis('off'); axes[0,col].set_title(f"Hard (σ={sig})",fontsize=8)
        axes[1,col].imshow(rs,cmap='gray',origin='lower'); axes[1,col].axis('off'); axes[1,col].set_title(f"ML soft (σ={sig})",fontsize=8)
    axes[0,0].set_ylabel("Hard",fontsize=9); axes[1,0].set_ylabel("ML soft",fontsize=9)
    plt.suptitle("Hard vs ML reconstruction at increasing noise",fontsize=10); plt.tight_layout(); display(fig2img(fig))

def iterative_refinement():
    it_sl=IntSlider(  value=3,  min=1,  max=8,   step=1,  description="Iterations",  style=_sty,layout=_sl3)
    bl_sl=FloatSlider(value=5.0,min=0.0,max=15.0,step=1.0,description="Init. blur σ", style=_sty,layout=_sl3)
    si_sl=FloatSlider(value=2.0,min=0.0,max=10.0,step=0.5,description="Noise σ",      style=_sty,layout=_sl3)
    rd_dd=Dropdown(options=['Q (oracle)','Blurred Q','Circle','P (wrong)','O (wrong)'],value='Blurred Q',description="Start ref:",style=_sty)
    out=Output()
    def _u(_=None):
        ta,imgs=simulate_images(Q,si_sl.value,100,seed=42); n=rd_dd.value; bl=bl_sl.value
        if n=='Blurred Q': ref=low_pass_filter(Q,sigma=bl)
        elif n=='Q (oracle)': ref=Q.copy()
        elif n=='Circle': ref=make_circle(64)
        elif n=='P (wrong)': ref=make_letter('P',64)
        else: ref=make_letter('O',64)
        history=[ref.copy()]; ccs=[]
        for _ in range(it_sl.value):
            ca,rs=rotate_reference(ref,delta_angle=5); est=align_images(imgs,rs,ca)
            ref=reconstruct(imgs,est); history.append(ref.copy())
            ccs.append(float((ref*Q).mean()/(np.sqrt((ref**2).mean()*(Q**2).mean())+1e-10)))
        ncols=min(len(history),6); fig,axes=plt.subplots(1,ncols+1,figsize=(3*(ncols+1),3.5))
        show(Q,axes[0],title="True A")
        for col,(im,lab) in enumerate(zip(history[:ncols],['Init']+[f'Iter {i+1}' for i in range(ncols-1)]),1):
            axes[col].imshow(im,cmap='gray',origin='lower'); axes[col].axis('off')
            axes[col].set_title(lab if col==1 else f"{lab}\nCC={ccs[col-2]:.2f}",fontsize=8)
        plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [it_sl,bl_sl,si_sl,rd_dd]: w.observe(_u,names='value')
    display(VBox([rd_dd,bl_sl,si_sl,it_sl,out])); _u()

def classification():
    K_sl=IntSlider(  value=2,  min=2,  max=3,  step=1,  description="# classes K", style=_sty,layout=_sl3)
    np_sl=IntSlider( value=40, min=10, max=100,step=5,   description="Images/class",style=_sty,layout=_sl3)
    si_sl=FloatSlider(value=1.5,min=0.0,max=6.0,step=0.5,description="Noise σ",    style=_sty,layout=_sl3)
    out=Output()
    def _u(_=None):
        K=K_sl.value; n_per=np_sl.value; sig=si_sl.value; letters=_CLS_LETTERS[:K]
        rng=np.random.default_rng(42); all_imgs,true_labels=[],[]
        for k,lt in enumerate(letters):
            idx=rng.integers(0,len(_CLS_CAND),n_per)
            noisy=_CLS_STACKS[lt][idx]+rng.standard_normal((n_per,64,64))*sig
            all_imgs.append(noisy); true_labels.extend([k]*n_per)
        all_imgs=np.concatenate(all_imgs); true_labels=np.array(true_labels)
        shuf=rng.permutation(len(all_imgs)); all_imgs=all_imgs[shuf]; true_labels=true_labels[shuf]
        rsa=np.concatenate([_CLS_STACKS[lt] for lt in letters]); ncp=len(_CLS_CAND); na=all_imgs.shape[0]
        cc=(all_imgs.reshape(na,64*64).astype(float)@rsa.reshape(len(rsa),64*64).astype(float).T)/(64*64)
        bg=cc.argmax(1); pc=bg//ncp; bai=bg%ncp; acc=(pc==true_labels).mean()*100
        class_recons=[]
        for k in range(K):
            mk=(pc==k); rk=np.zeros((64,64)); cnt=0
            for i in np.where(mk)[0]: rk+=nd_rotate(all_imgs[i],-_CLS_CAND[bai[i]],reshape=False); cnt+=1
            class_recons.append(rk/max(cnt,1))
        fig,axes=plt.subplots(1,K+2,figsize=(3.5*(K+2),3.5))
        show(all_imgs.mean(0),axes[0],title="Naive average\n(blurry)")
        for k,(rk,lt) in enumerate(zip(class_recons,letters)):
            axes[k+1].imshow(rk,cmap='gray',origin='lower'); axes[k+1].axis('off')
            axes[k+1].set_title(f"Class {k+1}: {lt}\n({(pc==k).sum()} imgs, acc={acc:.0f}%)",fontsize=8)
        axes[K+1].axis('off')
        plt.suptitle(f"2D Classification: K={K}, σ={sig}, {n_per} imgs/class",fontsize=9); plt.tight_layout()
        with out: clear_output(wait=True); display(fig2img(fig))
    for w in [K_sl,np_sl,si_sl]: w.observe(_u,names='value')
    display(VBox([K_sl,np_sl,si_sl,out])); _u()

print("Setup complete.")


# Practical: Single-Particle Analysis

This practical covers the core algorithmic steps of single-particle analysis: alignment, reconstruction, and classification.

> **To start the practicals:** click **Run All** (⏭) in the toolbar above.

---

## Part 0 – The test image

In [ ]:
show_test_image()

---

## Simulating cryo-EM images

The interactive below shows a single simulated particle image. The image formation model is:

$$
X = R^\theta A + \sigma G
$$

Drag the **angle** slider to rotate the particle and the **σ** slider to add noise.

In [ ]:
single_image()

---

## Part 1 – Choosing an initial reference

The alignment algorithm needs a starting reference to compare images against. Three strategies are common:

| Reference | Advantage | Risk |
|-----------|-----------|------|
| Low-pass filtered true structure | Fast convergence | Only works if structure is already known (circular reasoning) |
| Featureless circle | No reference bias | Slow convergence; poor angular discrimination |
| Random noise / another structure | Quick to obtain | Risk of converging to wrong answer (model bias) |

In practice, a featureless circle or a low-pass-filtered version of a prior reconstruction is used. The interactive below lets you choose the starting reference and see how much angular discrimination it provides for a single noisy image.

In [ ]:
reference_selection()

---

## Part 2 – Alignment algorithm

For a set of $N$ images $\{X_i\}$, we align each to the reference by searching over discrete candidate angles $\Delta\theta$ apart. For each image we rotate the reference to all candidate angles and pick the best-matching one:

$$
\hat\theta_i = \arg\max_{\theta_j} \bigl(X_i \cdot R^{\theta_j} A_\text{ref}\bigr)
$$

After estimating all angles, the scatter plot of true vs estimated angles reveals how well alignment worked. A tight diagonal indicates good alignment; spread indicates confusion.

```{admonition} Task
:class: note
**Part 2, step 1:** Implement the alignment loop in the code cell below. For each image, compute the correlation against every rotated reference and record the best angle. Run the cell to see the scatter plot of true vs estimated angles.
```

In [ ]:
alignment_demo()

---

## Part 3 – Reconstruction

Given estimated angles, rotating each image back and averaging gives the reconstruction:

$$
A \leftarrow \frac{1}{N} \sum_{i=1}^N (R^{\hat\theta_i})^{-1} X_i
$$

```{admonition} Task
:class: note
**Part 3:** Complete the reconstruction: for each image, rotate it back by $-\hat\theta_i$ and sum all rotated images. Divide by $N$ to get the average.
```

In [ ]:
reconstruction_result()

---

## Interactive reconstruction

The widget below runs the complete pipeline — from noisy images to reconstruction — with adjustable parameters. It also shows the scatter of estimated vs true angles and the correlation between reconstruction and true image as a quality metric.

In [ ]:
reconstruction_explorer()

---

## Bonus 1 – Adding translation

So far we assumed particles are perfectly centred. In reality, particle positions within the extracted box vary. The model extends to:

$$
X_i = T^{t_i}(R^{\theta_i} A) + \sigma G_i
$$

where $T^{t_i}$ is a translation by vector $\mathbf{t}_i = (dx_i, dy_i)$. We can handle integer-pixel translations by rolling the image array. The search grid is now $N_\theta \times N_{dx} \times N_{dy}$, so runtime increases by a factor of $(2 t_\text{max}+1)^2$.

In [ ]:
translation_demo()

In [ ]:
translation_alignment()

---

## Bonus 2 – Bayesian maximum-likelihood alignment

Hard assignment to the single best-fitting angle ignores alignment uncertainty. The **maximum-likelihood** approach instead computes a probability for each angle:

$$
P(X_i \mid \theta_j, A) \propto \exp\!\left[-\frac{\|X_i - R^{\theta_j}A\|^2}{2\sigma^2}\right]
$$

The **soft** reconstruction then uses all angles weighted by their posterior:

$$
w_{ij} = \frac{P(X_i \mid \theta_j, A)}{\sum_{j'} P(X_i \mid \theta_{j'}, A)}, \qquad
A \leftarrow \sum_i \sum_j w_{ij} (R^{\theta_j})^{-1} X_i
$$

At high SNR the weights collapse onto the single best angle (recovering hard assignment); at low SNR they spread across many angles, effectively averaging out alignment noise.

In [ ]:
ml_comparison()

---

## Bonus 3 – Iterative refinement

The previous examples used the initial reference for all alignment. A better strategy is to use the reconstruction from one round as the reference for the next:

1. Start from initial reference $A^{(0)}$ (e.g. blurred Q or a circle)
2. Align images to $A^{(k)}$ → estimate $\hat\theta_i^{(k)}$
3. Reconstruct $A^{(k+1)}$ from aligned images
4. Repeat from step 2

The quality of the reconstruction typically improves each iteration. The widget below lets you choose the starting reference and the number of iterations.

In [ ]:
iterative_refinement()

---

## Bonus 4 – 2D Classification

Real datasets contain multiple particle types (different proteins, conformations, or viewing directions). **2D classification** assigns each image to one of $K$ classes simultaneously with the alignment:

1. Maintain $K$ references $\{A_1, \ldots, A_K\}$
2. For each image, find the best (class, angle) pair by correlation
3. Reconstruct each class from the images assigned to it
4. Update references and iterate

The interactive below mixes $K$ letters (Q, P, R) and attempts to recover them. The classification accuracy depends on the noise level and the number of images per class.

In [ ]:
classification()

---

## Summary

| Topic | Key concept |
|-------|-------------|
| Observation model | $X_i = R^{\theta_i}A + \sigma G_i$ |
| Reference selection | Circular, blurred prior, or oracle; avoid reference bias |
| Alignment | Maximise correlation $\langle X_i, R^{\theta_j}A\rangle$ over discrete candidates |
| Reconstruction | Average back-rotated images |
| Model bias | Wrong reference → biased reconstruction; correct with iterative refinement |
| ML alignment | Soft weights via $\exp(-\|X_i - R^\theta A\|^2/2\sigma^2)$; better at low SNR |
| Iterative refinement | Use reconstruction as new reference; converges toward true structure |
| Classification | $K$-class joint alignment+assignment; separates heterogeneous datasets |